# 01. Download BTS On-Time monthly zips

Download US BTS Airline On-Time Performance (Reporting Carrier) monthly PREZIP files into `data/raw/bts_monthly/`.

**Default for this project:** full year **2025** (12 files). Skips zips that already exist and look complete (>1 MB).

Open this notebook with the working directory set to `airport-authority` (or keep it under `notebooks/` and the path cell will resolve `ROOT` automatically).

In [ ]:
from __future__ import annotations

import time
import urllib.error
import urllib.request
from pathlib import Path

# Resolve project root whether the kernel cwd is airport-authority/ or notebooks/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUT_DIR = ROOT / "data" / "raw" / "bts_monthly"
URL_TMPL = (
    "https://transtats.bts.gov/PREZIP/"
    "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)

## Parameters

Edit these cells instead of CLI flags.

In [ ]:
START_YEAR = 2025
END_YEAR = 2025
FORCE = False          # True = re-download even if file exists
SLEEP_SECONDS = 0.5    # pause between downloads

assert START_YEAR <= END_YEAR, "START_YEAR must be <= END_YEAR"

## Download helper

In [ ]:
def download_one(year: int, month: int, force: bool = False) -> str:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    dest = OUT_DIR / f"On_Time_{year}_{month:02d}.zip"
    url = URL_TMPL.format(year=year, month=month)

    if dest.exists() and dest.stat().st_size > 1_000_000 and not force:
        return f"SKIP {dest.name} ({dest.stat().st_size:,} bytes)"

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; airport-authority-capstone/1.0)"},
    )
    tmp = dest.with_suffix(".partial")
    try:
        with urllib.request.urlopen(req, timeout=300) as resp, tmp.open("wb") as out:
            while True:
                chunk = resp.read(1024 * 256)
                if not chunk:
                    break
                out.write(chunk)
        size = tmp.stat().st_size
        if size < 1_000_000:
            tmp.unlink(missing_ok=True)
            return f"FAIL {dest.name} too small ({size} bytes) url={url}"
        tmp.replace(dest)
        return f"OK   {dest.name} ({size:,} bytes)"
    except urllib.error.HTTPError as e:
        tmp.unlink(missing_ok=True)
        return f"FAIL {dest.name} HTTP {e.code}"
    except Exception as e:
        tmp.unlink(missing_ok=True)
        return f"FAIL {dest.name} {type(e).__name__}: {e}"

## Run download

In [ ]:
jobs = [(y, m) for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)]
print(f"Downloading {len(jobs)} monthly zips into {OUT_DIR}")

ok = skip = fail = 0
for i, (year, month) in enumerate(jobs, start=1):
    msg = download_one(year, month, force=FORCE)
    print(f"[{i}/{len(jobs)}] {msg}")
    if msg.startswith("OK"):
        ok += 1
    elif msg.startswith("SKIP"):
        skip += 1
    else:
        fail += 1
    time.sleep(SLEEP_SECONDS)

print(f"Done. ok={ok} skip={skip} fail={fail}")
if fail:
    raise RuntimeError(f"{fail} download(s) failed")